# From basic router → semantic router → self-learning

Three stages, in order of how a user adopts the library.

1. **Basic router** — you tell it which models to use, in what order. Fallback and load-balancing.
2. **Semantic router** — pre-trained weights pick the best model *per prompt* based on clusters + per-model error profiles. No rules from you.
3. **Self-learning** — as your app generates traces, the router's view of "which model is good at which cluster" updates. Feed in new data → get a router that adapts.

In [1]:
import os, sys, numpy as np
SRC = os.path.abspath(os.path.join(os.getcwd(), ".."))
if SRC not in sys.path: sys.path.insert(0, SRC)

import lunar_router as lr
print(f"lunar-router {lr.__version__}")

lunar-router 0.2.0


## 1. Basic router — rules you write yourself

`lr.Router` is the LiteLLM-style router: you define aliases (logical names like `smart`, `fast`) mapped to concrete provider deployments, plus a fallback chain. It handles retries and failover.

This is the *floor* — useful for availability, not for cost optimization. You still decide which model is smart or fast.

In [2]:
router_basic = lr.Router(
    model_list=[
        {"model_name": "smart", "model": "openai/gpt-4o"},
        {"model_name": "smart", "model": "anthropic/claude-sonnet-4-6"},  # redundancy
        {"model_name": "fast",  "model": "groq/llama-3.1-8b-instant"},
    ],
    fallbacks=[{"smart": ["deepseek/deepseek-chat"]}],
    strategy="round-robin",
    num_retries=2,
)
print("deployments:", {k: [d.model for d in v] for k, v in router_basic._deployments.items()})
print("fallbacks:  ", router_basic._fallbacks)
print()
print("usage (pseudocode):")
print('  resp = router_basic.completion(model="smart", messages=[...])')
print('  # tries gpt-4o → claude-sonnet-4-6 → deepseek on failure')

deployments: {'smart': ['openai/gpt-4o', 'anthropic/claude-sonnet-4-6'], 'fast': ['groq/llama-3.1-8b-instant']}
fallbacks:   {'smart': ['deepseek/deepseek-chat']}

usage (pseudocode):
  resp = router_basic.completion(model="smart", messages=[...])
  # tries gpt-4o → claude-sonnet-4-6 → deepseek on failure


**Limitation:** the basic router has no notion of *"this prompt would run fine on the cheap model, that one really needs gpt-4o"*. You'd route everything through whatever alias you hardcoded. The semantic router fixes exactly that.

---

## 2. Semantic router — pre-trained, picks per prompt

`lr.load_router()` downloads pre-trained weights (on first run) that encode:
- **100 semantic clusters** of prompts (learned from a benchmark)
- A **per-model error profile** over those clusters (how often each model fails on each cluster)

At route time: prompt → embedding → cluster → argmin(error + λ·cost) across models. You don't write any rules.

In [3]:
# Use the Python backend so we can introspect the learned profiles below.
# (The Go backend is faster and is what you'd use in production — same API.)
router = lr.load_router(engine="python", cost_weight=0.5, verbose=False)

print(f"backend:   {type(router).__name__}")
print(f"models:    {len(router.registry.get_model_ids())}")
print(f"clusters:  {router.cluster_assigner.num_clusters}")
print(f"embedder:  {type(router.embedder).__name__} (dim={router.embedder.dimension})")

/home/ubuntu/ml-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1570.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


backend:   UniRouteRouter
models:    10
clusters:  100
embedder:  PromptEmbedder (dim=384)


/home/ubuntu/lunar-router/lunar_router/core/embeddings.py:260: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self.model.get_sentence_embedding_dimension()


In [ ]:
# Route three very different prompts — router picks different models per cluster.
prompts = [
    "What is the capital of France?",
    "Write a Python function that reverses a linked list.",
    "Draft a slogan for an eco-friendly coffee brand.",
]

for p in prompts:
    d = router.route(p)
    print(f"  cluster {d.cluster_id:>3}  →  {d.selected_model:<28}  err={d.expected_error:.3f}  {p[:45]}")

The easy trivia went to `ministral-3b` (cheap, accurate enough). The code went to `gpt-4o` (the cluster has high error for small models). The creative one fell wherever the pre-trained profile said it should.

---

## 3. Self-learning — traces reshape the routing

The semantic router ships with error estimates from a *benchmark*. Your real production traffic is different. When your app runs, you capture traces (prompt, model used, was-it-good, cost, latency); those traces compute a **production-observed error profile** per model, per cluster; then you **blend** production and benchmark profiles to get a router that reflects your actual workload.

Primitive used: `TraceToTraining` + `blend_with_profiles(alpha=...)` where `alpha` is how much to trust production over the benchmark.

Below we simulate this: pick a prompt, see the current pick, feed in fake traces where that pick *failed badly* in production, rebuild the router, and see the routing shift.

In [ ]:
# Pick a focus prompt
FOCUS = "Explain the difference between a process and a thread."

d_before = router.route(FOCUS)
cluster  = d_before.cluster_id
picked_before = d_before.selected_model

# Snapshot: what the pre-trained profile says about each model on this cluster
top = sorted(
    router.registry.get_model_ids(),
    key=lambda m: router.registry.get(m).get_cluster_error(cluster) + 0.5 * router.registry.get(m).cost_per_1k_tokens,
)

print(f"prompt  : {FOCUS!r}")
print(f"cluster : {cluster}")
print(f"picked  : {picked_before}  (err={d_before.expected_error:.3f})")
print()
print(f"top 5 models on cluster {cluster} by cost-weighted score:")
for m in top[:5]:
    p = router.registry.get(m)
    err  = p.get_cluster_error(cluster)
    cost = p.cost_per_1k_tokens
    print(f"  {m:<28}  err={err:.3f}  cost=${cost:.5f}/1k  → score {err + 0.5*cost:.4f}")

In [ ]:
# Pretend production data tells a different story: the model the router
# currently picks fails badly on this cluster. Fabricate 20 traces — 15 errors,
# 5 successes — all on (picked_before, cluster).
from lunar_router.feedback.trace_to_training import TraceToTraining, TraceRecord

bad_traces = [
    TraceRecord(
        request_id=f"req-bad-{i}",
        selected_model=picked_before,
        cluster_id=cluster,
        is_error=(i < 15),          # 15/20 failed
        latency_ms=900.0,
        total_cost_usd=0.0003,
    )
    for i in range(20)
]

# ...and some solid traces for a small rival on the same cluster, so
# there's an obvious alternative for the router to switch to.
rival = next(m for m in router.registry.get_model_ids() if m != picked_before)
good_traces = [
    TraceRecord(
        request_id=f"req-good-{i}",
        selected_model=rival,
        cluster_id=cluster,
        is_error=False,             # 20/20 succeeded
        latency_ms=700.0,
        total_cost_usd=0.0001,
    )
    for i in range(20)
]

print(f"simulated traces: {len(bad_traces)} (error-heavy on '{picked_before}'),",
      f"{len(good_traces)} (clean on rival '{rival}')")

In [ ]:
# Run the feedback pipeline: traces → psi updates → blended profiles → new router.
num_clusters  = router.cluster_assigner.num_clusters
converter = TraceToTraining(num_clusters=num_clusters, latency_threshold_ms=30_000)
converter.add_traces(bad_traces + good_traces)

current_profiles = [router.registry.get(mid) for mid in router.registry.get_model_ids()]
new_profiles = converter.blend_with_profiles(current_profiles, alpha=0.5)  # 50% prod, 50% benchmark

# Rebuild the router with updated profiles
from lunar_router.models.llm_registry import LLMRegistry
from lunar_router.router.uniroute import UniRouteRouter

new_registry = LLMRegistry.from_profiles(new_profiles) if hasattr(LLMRegistry, "from_profiles") else LLMRegistry()
if not hasattr(LLMRegistry, "from_profiles"):
    for p in new_profiles:
        new_registry.register(p)

router_v2 = UniRouteRouter(
    embedder=router.embedder,
    cluster_assigner=router.cluster_assigner,
    registry=new_registry,
    cost_weight=router.cost_weight,
)

print(f"built router v2 — blended with alpha=0.5 (half-trust in production)")

In [ ]:
# Route the SAME focus prompt on the updated router and compare.
d_after = router_v2.route(FOCUS)

err_before = router.registry.get(picked_before).get_cluster_error(cluster)
err_after  = router_v2.registry.get(picked_before).get_cluster_error(cluster)
rival_before = router.registry.get(rival).get_cluster_error(cluster)
rival_after  = router_v2.registry.get(rival).get_cluster_error(cluster)

print(f"{'':<22} {'before':>10}   {'after':>10}")
print(f"{'selected model':<22} {picked_before:>10}   {d_after.selected_model:>10}")
print(f"err({picked_before[:18]:<18}) {err_before:>10.3f}   {err_after:>10.3f}  ← degraded (prod traces were bad)")
print(f"err({rival[:18]:<18}) {rival_before:>10.3f}   {rival_after:>10.3f}  ← improved (prod traces were clean)")
print()
shifted = (d_after.selected_model != picked_before)
print(f"routing shifted?  {'YES — router learned from traces' if shifted else 'no (benchmark still dominates at alpha=0.5; try alpha=0.8)'}")

## What just happened

```
production traces  ─────┐
                        ▼
                  TraceToTraining
                  (per-cluster error rates)
                        │
                        ▼
             blend_with_profiles(alpha)
          new_psi = α·prod + (1-α)·benchmark
                        │
                        ▼
                  new LLMRegistry
                        │
                        ▼
                  new UniRouteRouter  ← routing shifts
```

- **`alpha`** controls trust in production. Start small (0.1–0.3) and raise as you get more traces.
- In production this loop runs periodically (nightly cron, or after N new traces) via `IncrementalUpdater` which also persists updated weights to disk.
- **`DriftDetector`** catches a harder failure mode: when production prompts stop landing near any cluster centroid — signal to retrain clusters, not just profiles.

The full pipeline that turns captured traces into a **new distilled student model** (the wedge) builds on top of this same trace store — same data, different use: instead of re-weighting existing models, you train a cheap one that matches the teacher on your clusters.